In [ ]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

<table align="left">
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/colab/import/https:%2F%2Fraw.githubusercontent.com%2Fgooglemaps-samples%2Finsights-samples%2Fmain%2Fcustom_satellite_embeddings%2Fnotebooks%2Fee_classification%2FCustom_Satellite_Embeddings_Classification.ipynb?utm_source=custom_satellite_embeddings_notebooks">
      <img width="32px" src="https://lh3.googleusercontent.com/JmcxdQi-qOpctIvWKgPtrzZdJJK-J3sWE1RsfjZNwshCFgE_9fULcNpuXYTilIR2hjwN" alt="Google Cloud Colab Enterprise logo"><br> Open in Colab Enterprise
    </a>
  </td>
</table>

# Custom Satellite Embeddings Supervised and Unsupervised Classification

Custom satellite embeddings imagery is delivered as [Cloud Optimized Geotiffs](https://cogeo.org/) (COGs) in [Google Cloud Storage](https://cloud.google.com/storage). This guide assumes that [Google Earth Engine](https://earthengine.google.com) `ImageCollection`s of COG-backed images are already created.  See the Custom Satellite Embeddings Data Exploration guide for more information about the embeddings and COGs in Earth Engine.

In this guide, supervised and unsupervised classifications are fitted to a time series of six monthly satellite embeddings images in an agricultural area of the California, USA central valley. Use these methods to indentify areas of agricultural change or temporal trajectories of particular crop types. An animated visualizion is produced to illustrate change over time.

## Earth Engine Authentication and Initialization

To use Earth Engine, you must have a Cloud Project with the Earth Engine API activated.

In [ ]:
import ee
import geemap

from IPython.display import Image, display
from pprint import pprint

In [ ]:
# Your project with the Earth Engine API activated.
PROJECT = "YOUR-PROJECT"

In [ ]:
# Initialize Earth Engine (Assumes you have already authenticated via gcloud)
try:
  ee.Initialize(project=PROJECT)
except Exception:
  ee.Authenticate()
  ee.Initialize(project=PROJECT)

## Load the satellite embeddings `ImageCollection`

This sample is a time series of monthly images, centered on the 15th of the month, and spanning six months between 2025-11-15 and 2026-05-15.  The data covers an area of ~18,000 hectares of mostly cropland in California.

In [ ]:
meloland_california_2026_crop_cycles = ee.ImageCollection(
    "projects/mldp-trusted-testers-external/assets/alphaearth_foundations/preview/demos/meloland_california_2026_crop_cycles")

## Spatial mosaicking

Since Satellite embeddings are tiled, first aggregate the collection spatially, such that each image in the collection represents one time.  Here `mosaic()` is used to spatially aggregate the imagery.

In [ ]:
# Spatial mosaicking.
time_starts = meloland_california_2026_crop_cycles.aggregate_histogram('system:time_start')

def mosaic_function(time):
  time_start = ee.Number.parse(time)
  date = ee.Date(time_start)
  filtered = meloland_california_2026_crop_cycles.filterDate(date)
  # Transfer time and geographic properties.
  return filtered.mosaic().set('system:time_start', time_start).clip(filtered.geometry(100))

c = ee.ImageCollection.fromImages(time_starts.keys().map(mosaic_function))

## Unsupervised Classification

### Stack the entire time series in a single image

Stack each of the six 64-band images into a single image.  Use clusters in this space to represent temporal trajectories of different crops through time.  Earth Engine has native methods for sampling and clustering. Here Weka's k-means clusterer with seven clusters is used.

In [ ]:
# K-means on the 6x64D stack.
k = 7
n = 1000

# Temporal trajectory clusters.
big_stack = c.toBands()
training = big_stack.sample(
    region=c.first().geometry(),
    numPixels=n,
    scale=10,
)
training = training.filter(ee.Filter.notNull(big_stack.bandNames()))

clusterer = ee.Clusterer.wekaKMeans(k).train(training, big_stack.bandNames())
clustered = big_stack.cluster(clusterer)

Use `geemap` to visualize the results interactively.

In [ ]:
Map = geemap.Map(center=(32.8287, -115.45131), zoom=13)
roygbiv = ["FFFFFF", "#F60000", "#FF8C00", "#FFEE00", "#4DE94C", "#3783FF", "#4815AA"]
cluster_vis = {"min": 0, "max": k-1, "palette": roygbiv}
Map.addLayer(clustered, cluster_vis, 'clustered')
Map

Or request a thumbnail URL, which you can display using an IPython widget, download with `curl`, copy/paste, etc.  These URLs are temporary and should not be used for long term access.

In [ ]:
thumb_clusters_vis = cluster_vis.copy()
thumb_clusters_vis["dimensions"] = "512x512"
thumb_url = clustered.clip(c.first().geometry()).getThumbUrl(thumb_clusters_vis)
display(Image(url=thumb_url))

## Supervised Classification

### Use the annual USDA Cropland Data Layer (CDL) as reference labels

For reference labels, overlay the annual CDL crop type map with annual satellite embeddings to get a training dataset.  Train a Random Forest classifier to predict the top five crops in the specified region.  Classify every image in the custom satellite embeddings time series, then make an animated thumbnail of the time series of classifications.

In [ ]:
# CDL classification for 2025.
cdl = ee.ImageCollection("USDA/NASS/CDL")
cdl_2025 = cdl.filter(ee.Filter.calendarRange(2025, 2025, 'year')).first()

# Sample the CDL and annual embeddings for the same year.
aef = ee.ImageCollection("GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL")
aef_2025 = aef.filter(ee.Filter.calendarRange(2025, 2025, 'year')).mosaic()
sample = aef_2025.addBands(cdl_2025).sample(
  region=c.first().geometry(),
  numPixels=n,
  scale=10,
)

# Dominant classes only, the top five.
cropland = 'cropland'
histo = ee.Dictionary(sample.aggregate_histogram(cropland))
keys = histo.keys().sort(histo.values())
top_classes = keys.slice(keys.length().subtract(5))

def remap_class(f: ee.Feature) -> ee.Feature:
  """Remap the top five classes to consecutive integers, all else to 0."""
  old_class = f.get(cropland)
  new_class = top_classes.indexOf(ee.String(old_class)).add(1)
  return f.set(cropland, new_class)

training = sample.map(remap_class)
rf = ee.Classifier.smileRandomForest(10).train(training, cropland, aef_2025.bandNames())

Display an animation of the classificatons.

In [ ]:
rf_vis = {"min": 0, "max": 5, "palette": roygbiv[0:6]}
# Classify and visualize each image in the time series.
classified = c.map(lambda image: image.classify(rf).visualize(**rf_vis))
# Get a URL for the animation and use an IPython widget to display it.
anim_url = classified.getVideoThumbURL({ "framesPerSecond": 3, "dimensions": "512x512" })
display(Image(url=anim_url))

## What's next?

Explore [other Custom Satellite Embeddings samples](https://developers.devsite.corp.google.com/maps/documentation/custom-satellite-embeddings/samples).